### Import Dependencies

In [ ]:
import random
from qdrant_client import QdrantClient

import instructor
import json

from pydantic import BaseModel, Field

from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client
import tiktoken

In [ ]:
from dotenv import load_dotenv

load_dotenv("../../.env")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

### Load all data from Qdrant

In [ ]:
from qdrant_client.models import Record


HYBRID_SEARCH_COLLECTION_NAME = "Amazon-items-collection-01-hybrid-search"
all_points: list[Record] = []
next_offset = None
batch_size = 100

while True:
  points, next_offset = qdrant_client.scroll(
    collection_name=HYBRID_SEARCH_COLLECTION_NAME,
    limit=batch_size,
    offset=next_offset,
    with_payload=True,
    with_vectors=False
  )
  all_points.extend(points)
  if next_offset is None:
    break

len(all_points)

In [ ]:
all_points

### Split data into 2 groups
#### 50 items for synthetic question generation, 950 items to loop through and evaluate their relevance for previously generated synthetic questions

In [ ]:
rng = random.Random(42)
indices = list(range(len(all_points)))
indices

In [ ]:
rng.shuffle(indices)
indices

In [ ]:
sample_idx = set(indices[:50])
sample_idx

In [ ]:
sample_50 = [p for i, p in enumerate(all_points) if i in sample_idx]

In [ ]:
remaining_points = [p for i, p in enumerate(all_points) if i not in sample_idx]

### Generate synthetic questions

In [ ]:
from typing import TypedDict
from api.api.models import ItemPayload

ProductInfo = TypedDict("ProductInfo", {"id": str, "description": str})

product_infos_sample_50: list[ProductInfo] = [
    ProductInfo(
        id=data.parent_asin,
        description=data.preprocessed_description,
    )
    for data in [ItemPayload.model_validate(p.payload) for p in sample_50]
]

In [ ]:
product_infos_sample_50

In [ ]:
len(product_infos_sample_50)

In [ ]:
product_infos_remaining: list[ProductInfo] = [
    ProductInfo(
        id=data.parent_asin,
        description=data.preprocessed_description,
    )
    for data in [ItemPayload.model_validate(p.payload) for p in remaining_points]
]

In [ ]:
len(product_infos_remaining)

In [ ]:
SYSTEM_PROMPT = f"""
You are a test-data generator for a Retrieval-Augmented Generation (RAG) system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions about the products in stock by retrieving the most relevant products and grounding its answers in them.

## Your input
You will be given a sample of 50 products, each as an object with an `id` and a `text` description. This sample is part of a larger collection that will eventually contain 1000 products.

## Your task
Generate exactly 30 questions (MUST BE 30! NO LESS, NO MORE!!!) that a real customer of this e-shop might ask, split into three categories:

1. **Single-product (15 questions)** — answerable using exactly ONE product.
2. **Multi-product (10 questions)** — answerable using 2 to 3 products. Never more than 3.
3. **Unanswerable (5 questions)** — plausible customer questions that CANNOT be answered from any of the provided products.

## Constraints
- Write questions from the customer's point of view, in natural language.
- Refer to the items as "products". Never use the word "chunk".
- Keep questions specific. Even in the full 1000-product collection, a good question should be answerable from at most 4 products. Avoid broad or generic questions that a large number of products could satisfy.

## Products
{json.dumps(product_infos_sample_50, indent=2)}
"""

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
client = instructor.from_provider(
  "openai/gpt-5.4",
  mode=instructor.Mode.RESPONSES_TOOLS
)

In [ ]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
        },
    },
}

class SyntheticQuestionWithAnswerAndGrounding(BaseModel):
  reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
  question: str = Field(description="Suggested question.")
  chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")
  answer_example: str = Field(description="Suggested answer grounded in the context.")

class BatchSyntheticQuestionsWithAnswerAndGrounding(BaseModel):
  questions: list[SyntheticQuestionWithAnswerAndGrounding] = Field(description="List of synthetic questions", min_length=30, max_length=30)

In [ ]:
response, raw_response = client.create_with_completion(
    messages=[{"role": "user", "content": SYSTEM_PROMPT}],
    reasoning={"effort": "none"},
    response_model=BatchSyntheticQuestionsWithAnswerAndGrounding,
)

In [ ]:
response

In [ ]:
len(response.questions)


In [ ]:
response.questions

In [ ]:
for s_question in response.questions:
  print(s_question.chunk_ids)

In [ ]:
class SyntheticQuestion(TypedDict): 
  question_id: int
  question: str

synthetic_questions = [SyntheticQuestion(question_id=i, question=item.question) for i, item in enumerate(response.questions)]

In [ ]:
synthetic_questions

In [ ]:
SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions, indent=2)}

## Products
{json.dumps(product_infos_remaining[0:50], indent=2)}
"""


In [ ]:
print(SYSTEM_PROMPT_950)

### Count the number of tokens for the static prefix

In [ ]:
SYSTEM_PROMPT_STATIC_PART = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions, indent=2)}

## Products
"""


In [ ]:
def count_tokens(text: str) -> int:
  encoding = tiktoken.get_encoding("o200k_base")
  return len(encoding.encode(text))

In [ ]:
print(count_tokens(SYSTEM_PROMPT_STATIC_PART))

### Assign chunks to relevant questions for a single batch

In [ ]:
client = instructor.from_provider(
  "openai/gpt-5.4-mini",
  mode=instructor.Mode.RESPONSES_TOOLS
)

In [ ]:
class ResultOfMatchingChunksToQuestion(BaseModel):
  reasoning: str = Field(description="Reasoning why the question could be answered with the chunks.")
  question_id: int = Field(description="ID of the question.")
  chunk_ids: list[str] = Field(description="IDs of the chunks that could be used to answer the question.")

class BatchOfResultsOfMatchingChunksToQuestion(BaseModel):
  results: list[ResultOfMatchingChunksToQuestion] = Field(description="List of results of matching chunks to questions.")

In [ ]:
tmp_response, tmp_raw_response = client.create_with_completion(
    messages=[{"role": "user", "content": SYSTEM_PROMPT_950}],
    reasoning={"effort": "none"},
    response_model=BatchOfResultsOfMatchingChunksToQuestion,
)

In [ ]:
tmp_response.results

In [ ]:
for data in tmp_response.results:
  print(f"Question {data.question_id}, Chunks: {data.chunk_ids}")

In [ ]:
from openai.types.responses import Response


if not isinstance(tmp_raw_response, Response) or tmp_raw_response.usage is None:
        raise ValueError(f"Unexpected raw response: {type(tmp_raw_response)}")
tmp_raw_response.usage

In [ ]:
def get_relevant_chunks(synthetic_questions: list[SyntheticQuestion], remaining_points_batch: list[ProductInfo]) -> list[ResultOfMatchingChunksToQuestion]:
    SYSTEM_PROMPT_950 = f"""
You are a relevance annotator building ground-truth labels for a RAG retrieval system.

## System being tested
The RAG system is a shopping assistant for an e-shop. It answers customer questions by retrieving relevant products and grounding its answers in them.

## Your input
1. A list of customer questions, each with an `id`.
2. A list of 50 products, each with an `id` and a `text` description.

## Your task
For each question, identify which individual products could be used to answer it. Apply a strict, per-product test:

- Include a product ID only if that product ALONE is sufficient to answer the question on its own.
- Do NOT include products that only help in combination with others. Judge each product independently, not as a group.
- If no product independently answers the question, return an empty list.

## Questions
{json.dumps(synthetic_questions, indent=2)}

## Products
{json.dumps(remaining_points_batch, indent=2)}
"""

    response, _ = client.create_with_completion(
        messages=[{"role": "user", "content": SYSTEM_PROMPT_950}],
        reasoning={"effort": "none"},
        response_model=BatchOfResultsOfMatchingChunksToQuestion,
    )

    return response.results


In [ ]:
batches_of_results_matching_chunks_to_questions: list[list[ResultOfMatchingChunksToQuestion]] = []

for i in range(0, len(product_infos_remaining), 50):
  print(f"Working on batch {(i+50)/50}")

  batches_of_results_matching_chunks_to_questions.append(get_relevant_chunks(synthetic_questions, product_infos_remaining[i:i+50]))

batches_of_results_matching_chunks_to_questions





In [ ]:
for i, batch in enumerate(batches_of_results_matching_chunks_to_questions):
  print(f"Batch {i+1}")
  for result in batch:
    print(f"Question ID: {result.question_id}, Chunks: {result.chunk_ids}")
    
    

In [ ]:
question_id_to_asins_relevant: dict[int, list[str]] = {i: [] for i in range(30)}

In [ ]:
question_id_to_asins_relevant

In [ ]:
for i, batch in enumerate(batches_of_results_matching_chunks_to_questions):
  for result in batch:
    question_id_to_asins_relevant[result.question_id].extend(result.chunk_ids)
question_id_to_asins_relevant



In [ ]:
for i, data in enumerate(response.questions):
  question_id_to_asins_relevant[i].extend(data.chunk_ids)

In [ ]:
question_id_to_asins_relevant

In [ ]:
def get_description(parent_asin: str) -> str:
    points = qdrant_client.scroll(
        collection_name=HYBRID_SEARCH_COLLECTION_NAME,
        scroll_filter=Filter(
            must=[
                FieldCondition(key="parent_asin", match=MatchValue(value=parent_asin))
            ]
        ),
    )[0]

    if not points[0].payload:
        raise ValueError(f"No payload in point: {points[0].id}")

    validated_payload = ItemPayload.model_validate(points[0].payload)

    return validated_payload.preprocessed_description


In [ ]:
get_description("B09WQWZQQJ")

In [ ]:
def generate_golden_answer(question, relevant_chunks):

    class GoldenAnswer(BaseModel):
        answer: str = Field(description="The ideal reference answer for the question.")

    PROMPT = f"""
You are the shopping assistant of an e-shop. You are also being used to generate the ideal reference answer for a single customer question, to serve as ground truth for evaluating a RAG system.

## Context
A customer asked a question. A retrieval system has already gathered the products most relevant to it. Your job is to write the best possible answer a helpful shopping assistant could give, using ONLY those products.

## Inputs
- `question`: the customer's question.
- `products`: a list of relevant products, each with a `text` description. This may be empty or may not actually contain the answer.

## How to write the answer
- Ground every factual claim strictly in the provided product descriptions. Never invent products, prices, specs, availability, or details that are not present in the text.
- Answer in the voice of a friendly, concise shopping assistant speaking directly to the customer.
- Use only the information needed to answer well; don't dump every product if only some are relevant.
- If the products only partially answer the question, answer what you can and clearly state what information isn't available.
- If the products do not contain the information needed to answer at all (or the list is empty), do not guess. Respond the way a good assistant would when the shop doesn't carry the item or the info isn't available (e.g. politely say you couldn't find it), and do not fabricate an answer.

## Question
{json.dumps(question, indent=2)}

## Products
{json.dumps(relevant_chunks, indent=2)}"""

    response, raw_response = client.create_with_completion(
        messages=[{"role": "system", "content": PROMPT}],
        reasoning={"effort": "none"},
        response_model=GoldenAnswer,
    )

    return response.answer


In [ ]:
ReferenceDatasetItem = TypedDict("ReferenceDatasetItem", {"question": str, "ground_truth": str, "relevant_context_ids": list[str], "reference_descriptions": list[str]})
reference_dataset: list[ReferenceDatasetItem] = []

for i, data in enumerate(response.questions):
  print(f"Working on question {i+1}")

  asins_relevant_to_question = question_id_to_asins_relevant[i]

  descriptions: list[str] = []

  for asin in asins_relevant_to_question:
    try:
      descriptions.append(get_description(asin))
    except Exception as e:
      print(f"Error getting description for ASIN: {asin}")
      print(e)
      asins_relevant_to_question.remove(asin)

  golden_answer = generate_golden_answer(data.question, descriptions)

  reference_dataset.append({
    "question": data.question,
    "ground_truth": golden_answer,
    "relevant_context_ids": asins_relevant_to_question,
    "reference_descriptions": descriptions
  })

reference_dataset

### Create eval dataset in LangSmith

In [ ]:
ls_client = Client()

dataset_name = "rag-evaluation-dataset-extended"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name, description="RAG evaluation dataset for 1000 products"
)
for item in reference_dataset:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["ground_truth"],
            "relevant_context_ids": item["relevant_context_ids"],
            "reference_descriptions": item["reference_descriptions"],
        },
    )
